Mount Drive and Setup Paths

In [ ]:
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Set paths
pfa_path = "/content/drive/MyDrive/my_PFA/ImageBind"
results_dir = os.path.join(pfa_path, "results")
imagebind_path = os.path.join(pfa_path, "ImageBind")

# Create directories
os.makedirs(pfa_path, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

# Add to path
sys.path.insert(0, imagebind_path)

print("✓ Paths set up successfully")
print(f"PFA Path: {pfa_path}")
print(f"Results Path: {results_dir}")

Mounted at /content/drive
✓ Paths set up successfully
PFA Path: /content/drive/MyDrive/my_PFA/ImageBind
Results Path: /content/drive/MyDrive/my_PFA/ImageBind/results


Install ONLY Essential Dependencies

In [ ]:
# Install ONLY what's needed (no redundant installations)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pytorchvideo ftfy timm einops fvcore decord
!pip install -q openai-whisper librosa soundfile opencv-python matplotlib tqdm
!pip install -q numpy==1.26.4  # Stable version

print("✓ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 19.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

Setup ImageBind

In [ ]:
# Clone and install ImageBind (only if not already there)
if not os.path.exists(imagebind_path):
    !git clone https://github.com/facebookresearch/ImageBind.git {imagebind_path}

# Install ImageBind
!cd {imagebind_path} && pip install -e .

print("✓ ImageBind setup complete")

Cloning into '/content/drive/MyDrive/my_PFA/ImageBind/ImageBind'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 187 (delta 84), reused 54 (delta 53), pack-reused 67 (from 3)
Receiving objects: 100% (187/187), 2.65 MiB | 9.73 MiB/s, done.
Resolving deltas: 100% (92/92), done.
Obtaining file:///content/drive/MyDrive/my_PFA/ImageBind/ImageBind
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/pytorchvideo.git (to revision 6cdc929315aab1b5674b6dcf73b16ec99147735f) to /tmp/pip-install-482uu72_/pytorchvideo_408e5e42d79e46ac971fe77ff22a4439
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorchvideo.git /tmp/pip-install-482uu72_/pytorchvideo_408e5e42d79e46ac971fe77ff22a4439
  Running command git rev-parse -q --verify 'sha^6cdc929315aab1b5674b6dcf73b16ec99147735f'
  Running command git fetch -q https://github

Import Libraries and Load Models

In [ ]:
# ============================================
# FIXED: Import Libraries with NumPy Compatibility
# ============================================

import sys
import subprocess

# First, check NumPy version and reinstall if needed
import numpy as np
print(f"Current NumPy version: {np.__version__}")

# If NumPy is not 1.26.4, reinstall the correct version
if np.__version__ != '1.26.4':
    print("NumPy version mismatch. Reinstalling correct version...")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.4"])

    # Force reload NumPy
    import importlib
    importlib.reload(np)
    print(f"NumPy version after fix: {np.__version__}")

# Now import everything else
import torch
import numpy as np  # Re-import to be sure
import json
import whisper
import librosa
import soundfile as sf
import subprocess
import tempfile
from PIL import Image
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
from decord import VideoReader, cpu
import torch.nn as nn
import torch.nn.functional as F

# Import ImageBind
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

# Set device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {device}")

# Load ImageBind model
print("Loading ImageBind model...")
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(device)
print("✓ ImageBind loaded")

# Load Whisper model
print("Loading Whisper model...")
whisper_model = whisper.load_model("base")
print("✓ Whisper loaded")

# Verify no conflicts
print(f"\nFinal versions:")
print(f"  NumPy: {np.__version__}")
print(f"  PyTorch: {torch.__version__}")
print(f"  Whisper: {whisper.__version__ if hasattr(whisper, '__version__') else 'installed'}")

Current NumPy version: 2.0.2
NumPy version mismatch. Reinstalling correct version...
NumPy version after fix: 2.0.2


ModuleNotFoundError: No module named 'torchvision.transforms.functional_tensor'

Video Processing Functions

In [ ]:
def segment_video(video_path, window_size=2, stride=1):
    vr = VideoReader(video_path, ctx=cpu(0))
    fps = vr.get_avg_fps()
    duration = len(vr) / fps

    segments = []
    t = 0
    while t + window_size <= duration:
        segments.append((t, t + window_size))
        t += stride

    print(f"Video duration: {duration:.1f}s, Segments: {len(segments)}")
    return segments, fps, vr

def extract_frames(vr, start_time, end_time, fps, num_frames=8):
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    frame_indices = np.linspace(start_frame, end_frame - 1, num_frames, dtype=int)
    return vr.get_batch(frame_indices).asnumpy()

def extract_audio_segment(video_path, start_time, end_time, sr=16000):
    audio_file = video_path.replace('.mp4', '.wav')
    if not os.path.exists(audio_file):
        cmd = ['ffmpeg', '-i', video_path, '-vn', '-acodec', 'pcm_s16le',
               '-ar', str(sr), '-ac', '1', audio_file, '-y']
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    audio, _ = librosa.load(audio_file, sr=sr, offset=start_time, duration=end_time-start_time)
    return audio

def extract_transcript_with_timestamps(video_path, whisper_model):
    result = whisper_model.transcribe(video_path, word_timestamps=True, verbose=False)
    segments = []
    for segment in result['segments']:
        segments.append({
            'start': segment['start'],
            'end': segment['end'],
            'text': segment['text'].strip()
        })
    return segments, result['text']

def align_text_to_segments(transcript_segments, video_segments):
    aligned_texts = []
    for vid_path, vid_start, vid_end in video_segments:
        overlapping_text = []
        for trans_seg in transcript_segments:
            if not (trans_seg['end'] < vid_start or trans_seg['start'] > vid_end):
                overlapping_text.append(trans_seg['text'])
        segment_text = " ".join(overlapping_text).strip()
        aligned_texts.append(segment_text)
    return aligned_texts

print("✓ Video processing functions loaded")

IF-MMIN Module Definition

In [ ]:
class CMDLoss(nn.Module):
    """Central Moment Discrepancy loss"""
    def __init__(self, max_order=5):
        super().__init__()
        self.max_order = max_order

    def forward(self, H_a, H_v, H_t):
        loss = 0.0
        pairs = [('a', 'v'), ('a', 't'), ('v', 't')]

        for m1, m2 in pairs:
            H1 = eval(f'H_{m1}')
            H2 = eval(f'H_{m2}')

            mean_loss = torch.norm(H1.mean(dim=0) - H2.mean(dim=0), p=2)
            loss += mean_loss

            for k in range(2, self.max_order + 1):
                H1_centered = H1 - H1.mean(dim=0, keepdim=True)
                H2_centered = H2 - H2.mean(dim=0, keepdim=True)
                moment1 = torch.mean(H1_centered ** k, dim=0)
                moment2 = torch.mean(H2_centered ** k, dim=0)
                loss += torch.norm(moment1 - moment2, p=2)

        return loss / len(pairs)

class IFMMINModule(nn.Module):
    def __init__(self, embed_dim=1024, specific_dim=256, invariant_dim=384):
        super().__init__()

        # Specificity Encoders
        self.specificity_encoders = nn.ModuleDict({
            ModalityType.VISION: nn.Sequential(
                nn.Linear(embed_dim, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, specific_dim)
            ),
            ModalityType.AUDIO: nn.Sequential(
                nn.Linear(embed_dim, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, specific_dim)
            ),
            ModalityType.TEXT: nn.Sequential(
                nn.Linear(embed_dim, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, specific_dim)
            )
        })

        # Invariance Encoder
        self.invariance_encoder = nn.Sequential(
            nn.Linear(specific_dim * 3, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, invariant_dim)
        )

        self.invariance_split = nn.Linear(invariant_dim, specific_dim * 3)

        # IF-IM: 5 cascaded autoencoders
        self.if_im = nn.ModuleList([
            nn.Sequential(
                nn.Linear(specific_dim * 2 + invariant_dim, 256), nn.ReLU(),
                nn.Dropout(0.2), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, specific_dim)
            ) for _ in range(5)
        ])

        self.to_imagebind = nn.Linear(specific_dim, embed_dim)

    def forward_with_text(self, vision_emb, audio_emb, text_emb):
        h_v = self.specificity_encoders[ModalityType.VISION](vision_emb)
        h_a = self.specificity_encoders[ModalityType.AUDIO](audio_emb)
        h_t = self.specificity_encoders[ModalityType.TEXT](text_emb)

        h_concat = torch.cat([h_v, h_a, h_t], dim=1)
        H = self.invariance_encoder(h_concat)

        imagined_text = self._imagine_text(h_v, h_a, H)

        return {
            'h_t_real': h_t,
            'h_t_imagined': imagined_text,
            'H': H
        }

    def forward_without_text(self, vision_emb, audio_emb):
        h_v = self.specificity_encoders[ModalityType.VISION](vision_emb)
        h_a = self.specificity_encoders[ModalityType.AUDIO](audio_emb)

        h_t_prior = torch.zeros_like(h_v)
        h_concat = torch.cat([h_v, h_a, h_t_prior], dim=1)
        H = self.invariance_encoder(h_concat)

        imagined_text_specific = self._imagine_text(h_v, h_a, H)
        return self.to_imagebind(imagined_text_specific)

    def _imagine_text(self, h_v, h_a, H):
        h_av = torch.cat([h_v, h_a], dim=1)
        delta_z = None
        for i, ae in enumerate(self.if_im):
            if i == 0:
                input_feat = torch.cat([H, h_av], dim=1)
                delta_z = ae(input_feat)
            else:
                input_feat = torch.cat([H, delta_z], dim=1)
                delta_z = ae(input_feat)
        return delta_z

class IFMMINTrainer:
    def __init__(self, device):
        self.device = device
        self.if_mmin = IFMMINModule().to(device)
        self.cmd_loss = CMDLoss()
        self.optimizer = torch.optim.Adam(self.if_mmin.parameters(), lr=0.0002)

    def training_step(self, vision_emb, audio_emb, text_emb):
        self.optimizer.zero_grad()
        outputs = self.if_mmin.forward_with_text(vision_emb, audio_emb, text_emb)

        # Split H into H_v, H_a, H_t
        H_split = self.if_mmin.invariance_split(outputs['H'])
        H_v, H_a, H_t = H_split.chunk(3, dim=1)

        loss_cmd = self.cmd_loss(H_v, H_a, H_t)
        loss_img = F.mse_loss(outputs['h_t_imagined'], outputs['h_t_real'])

        total_loss = loss_cmd + loss_img

        total_loss.backward()
        self.optimizer.step()

        return {'total_loss': total_loss.item(), 'cmd_loss': loss_cmd.item(), 'img_loss': loss_img.item()}

    def imagine_text(self, vision_emb, audio_emb):
        with torch.no_grad():
            return self.if_mmin.forward_without_text(vision_emb, audio_emb)

    def save(self, path):
        torch.save(self.if_mmin.state_dict(), path)

    def load(self, path):
        self.if_mmin.load_state_dict(torch.load(path, map_location=self.device))

print("✓ IF-MMIN module defined")

Enhanced Processing Function with IF-MMIN

In [ ]:
def process_segment_with_ifmmin(vr, video_path, start_time, end_time, fps,
                                segment_text, ifmmin_trainer=None, num_frames=8):
    """Process segment with intelligent text handling using IF-MMIN"""

    # Extract frames
    frames = extract_frames(vr, start_time, end_time, fps, num_frames)

    with tempfile.TemporaryDirectory() as tmpdir:
        # Save frames
        frame_paths = []
        for i, frame in enumerate(frames):
            frame_path = os.path.join(tmpdir, f"frame_{i}.jpg")
            Image.fromarray(frame).save(frame_path)
            frame_paths.append(frame_path)

        # Extract and save audio
        audio = extract_audio_segment(video_path, start_time, end_time)
        audio_path = os.path.join(tmpdir, "audio.wav")
        sf.write(audio_path, audio, 16000)

        # Get vision and audio embeddings from ImageBind
        inputs = {
            ModalityType.VISION: data.load_and_transform_vision_data(frame_paths, device),
            ModalityType.AUDIO: data.load_and_transform_audio_data([audio_path], device)
        }

        with torch.no_grad():
            embeddings = model(inputs)

        vision_emb = embeddings[ModalityType.VISION].cpu().numpy()
        audio_emb = embeddings[ModalityType.AUDIO].cpu().numpy()

        # Handle text intelligently
        if segment_text:
            # Real text available
            text_inputs = {ModalityType.TEXT: data.load_and_transform_text([segment_text], device)}
            with torch.no_grad():
                text_emb = model(text_inputs)[ModalityType.TEXT].cpu().numpy()
            text_source = "real"
            text_weight = 1.0
        else:
            # No text - use IF-MMIN to imagine it
            if ifmmin_trainer is not None:
                vision_tensor = torch.from_numpy(vision_emb).float().to(device).unsqueeze(0)
                audio_tensor = torch.from_numpy(audio_emb).float().to(device).unsqueeze(0)
                imagined_text = ifmmin_trainer.imagine_text(vision_tensor, audio_tensor)
                text_emb = imagined_text.cpu().numpy().squeeze()
                text_source = "imagined"
                text_weight = 0.8
            else:
                # Fallback to zeros if no trainer
                text_emb = np.zeros_like(vision_emb)
                text_source = "zero_fallback"
                text_weight = 0.0

        # Weighted combination
        total_weight = 1.0 + 1.0 + text_weight
        combined_emb = (vision_emb + audio_emb + text_weight * text_emb) / total_weight

    return {
        'vision': vision_emb, 'audio': audio_emb, 'text': text_emb,
        'combined': combined_emb, 'text_source': text_source,
        'start': start_time, 'end': end_time
    }

print("✓ Enhanced processing function ready")

Load Previous Results

In [ ]:
def load_existing_results():
    """Load previously extracted embeddings if they exist"""
    import glob

    emb_files = glob.glob(os.path.join(results_dir, "*_embeddings.npz"))
    meta_files = glob.glob(os.path.join(results_dir, "*.json"))

    if emb_files and meta_files:
        latest_emb = max(emb_files, key=os.path.getctime)
        base_name = os.path.basename(latest_emb).replace('_embeddings.npz', '.json')
        latest_meta = os.path.join(results_dir, base_name)

        if os.path.exists(latest_meta):
            print(f"Loading: {os.path.basename(latest_emb)}")
            emb_data = np.load(latest_emb)
            with open(latest_meta, 'r') as f:
                metadata = json.load(f)

            results = []
            for i in range(len(emb_data['vision'])):
                results.append({
                    'vision_emb': emb_data['vision'][i],
                    'audio_emb': emb_data['audio'][i],
                    'text_emb': emb_data['text'][i],
                    'text': metadata['segments'][i]['text'] if i < len(metadata['segments']) else '',
                    'start': emb_data['times'][i][0],
                    'end': emb_data['times'][i][1]
                })

            print(f"✓ Loaded {len(results)} segments")
            return results, metadata.get('full_transcript', '')

    print("No previous results found")
    return None, None

# Try to load existing results
existing_results, existing_transcript = load_existing_results()

Train IF-MMIN on Your Data

In [ ]:
# Initialize trainer
ifmmin_trainer = IFMMINTrainer(device)

# Use existing results if available, otherwise process a video first
if existing_results is not None:
    results_to_use = existing_results
    print("Using existing results for training")
else:
    # Process a video to get training data
    video_path = "/content/drive/MyDrive/PFA/data/Obama_Yes_we_can.mp4"  # Update this path
    if os.path.exists(video_path):
        print("Extracting features for training...")
        segments, fps, vr = segment_video(video_path, window_size=2, stride=1)
        transcript_segments, full_text = extract_transcript_with_timestamps(video_path, whisper_model)
        segment_tuples = [(None, start, end) for start, end in segments]
        segment_texts = align_text_to_segments(transcript_segments, segment_tuples)

        results_to_use = []
        for idx, (start, end) in enumerate(tqdm(segments, desc="Extracting")):
            seg_data = process_segment_with_ifmmin(vr, video_path, start, end, fps,
                                                  segment_texts[idx], ifmmin_trainer=None)
            results_to_use.append(seg_data)
    else:
        raise FileNotFoundError(f"Video not found: {video_path}")

# Filter segments with real text for training
text_segments = [r for r in results_to_use if r.get('text') and r.get('text_source') == 'real']
print(f"\nFound {len(text_segments)} segments with real text for training")

if len(text_segments) >= 10:
    # Prepare training data
    vision_data = torch.tensor(np.array([r['vision'] for r in text_segments])).float().to(device)
    audio_data = torch.tensor(np.array([r['audio'] for r in text_segments])).float().to(device)
    text_data = torch.tensor(np.array([r['text'] for r in text_segments])).float().to(device)

    # Train
    print("\nTraining IF-MMIN...")
    epochs = 30
    for epoch in range(epochs):
        losses = ifmmin_trainer.training_step(vision_data, audio_data, text_data)
        if epoch % 5 == 0:
            print(f"Epoch {epoch}: Loss = {losses['total_loss']:.4f}")

    # Save
    model_path = os.path.join(results_dir, "ifmmin_trained.pth")
    ifmmin_trainer.save(model_path)
    print(f"✓ Model saved to {model_path}")
else:
    print(f"⚠ Not enough text segments ({len(text_segments)}). Need at least 10.")

Test on a New Video

In [ ]:
def test_on_new_video(video_path, ifmmin_trainer):
    """Test the IF-MMIN on a new video"""

    print(f"\nTesting on: {os.path.basename(video_path)}")

    # Get segments and transcript
    segments, fps, vr = segment_video(video_path, window_size=2, stride=1)
    transcript_segments, full_text = extract_transcript_with_timestamps(video_path, whisper_model)
    segment_tuples = [(None, start, end) for start, end in segments]
    segment_texts = align_text_to_segments(transcript_segments, segment_tuples)

    text_count = sum(1 for t in segment_texts if t)
    print(f"Segments with text: {text_count}/{len(segments)}")

    # Process with IF-MMIN
    results = []
    for idx, (start, end) in enumerate(tqdm(segments, desc="Processing")):
        seg_data = process_segment_with_ifmmin(
            vr, video_path, start, end, fps,
            segment_texts[idx], ifmmin_trainer
        )
        results.append(seg_data)

    # Show statistics
    sources = {}
    for r in results:
        sources[r['text_source']] = sources.get(r['text_source'], 0) + 1

    print("\nText sources:")
    for source, count in sources.items():
        print(f"  {source}: {count}")

    # Example: Show imagined embeddings for silent segments
    print("\nExample - Silent segments (imagined text embeddings):")
    silent_count = 0
    for r in results:
        if r['text_source'] == 'imagined' and silent_count < 3:
            energy = np.mean(np.abs(r['text']))
            print(f"  Segment {r['start']:.1f}s: energy = {energy:.4f}")
            silent_count += 1

    return results

# Test on a new video
test_video = "/content/drive/MyDrive/PFA/data/test_video.mp4"  # Update this path
if os.path.exists(test_video) and 'ifmmin_trainer' in locals():
    test_results = test_on_new_video(test_video, ifmmin_trainer)

Save Results with IF-MMIN

In [ ]:
def save_results_with_ifmmin(results, video_path, full_transcript, ifmmin_trainer):
    """Save results including text source information"""

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Save metadata
    metadata_file = os.path.join(results_dir, f"ifmmin_results_{timestamp}.json")
    output_data = {
        'video': video_path,
        'timestamp': datetime.now().isoformat(),
        'num_segments': len(results),
        'segments': [
            {
                'start': r['start'],
                'end': r['end'],
                'text': r.get('text', ''),
                'text_source': r['text_source']
            }
            for r in results
        ],
        'full_transcript': full_transcript
    }

    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2)

    # Save embeddings
    emb_file = os.path.join(results_dir, f"ifmmin_embeddings_{timestamp}.npz")
    np.savez_compressed(
        emb_file,
        vision=np.array([r['vision'] for r in results]),
        audio=np.array([r['audio'] for r in results]),
        text=np.array([r['text'] for r in results]),
        times=np.array([(r['start'], r['end']) for r in results]),
        text_source=np.array([r['text_source'] for r in results])
    )

    print(f"✓ Results saved to {results_dir}")
    return metadata_file, emb_file

# Save if you have results
if 'test_results' in locals():
    save_results_with_ifmmin(test_results, test_video, full_text, ifmmin_trainer)